# Sanity checks for our Database
We want to check that our csv files are perfectly synced with each other and that all listed images physically exist on the storage drive before starting our data pipeline.

This Python script runs a multi-step data integrity audit using `pandas` and `pathlib`. It safely isolates mismatched IDs, missing images, and untracked files without crashing interactive environments like Jupyter Notebooks.

### Key Implementation Details

* **Path Verification**: Confirms the physical existence of data directories and CSV files.
* **Schema Audit**: Prints out exact column names to assist debugging and validates that required configuration keys (`artikelnummer`, `image_name`) exist.
* **Data Cleaning**: Drops rows containing missing values (`NaN`) in key identifier columns.
* **Cross-CSV Analysis**: Uses mathematical set operations to isolate IDs that exist uniquely in one database file but not the other.
* **Disk-to-CSV Alignment**: Normalizes image filenames, checks extension formats, and matches file entries directly against physical files inside the `/images` folder.
* **Automated Log Export**: Generates local diagnostic `.txt` files inside `/debug/reportes_errores/` listing every specific failure point whenever a data mismatch occurs.

### File Outputs Created on Error
* `debug/reportes_errores/1_imagenes_faltantes_en_disco.txt` - Files requested by CSV but missing from disk.
* `debug/reportes_errores/2_imagenes_sobrantes_en_disco.txt` - Untracked image files cluttering the storage directory.
* `debug/reportes_errores/3_ids_solo_en_csv_mapping.txt` - Item IDs missing from the product typicality database.
* `debug/reportes_errores/4_ids_solo_en_csv_typicality.txt` - Item IDs missing from the image mapping database.


In [3]:
import pandas as pd
from pathlib import Path

def reporte_exhaustivo():
    # 1. Definir las rutas relativas
    base_dir = Path(".")
    csv_mapping_path = base_dir / "data" / "artikelnummer_to_image.csv"
    csv_typicality_path = base_dir / "data" / "article_typicality.csv"
    images_dir = base_dir / "images"
    
    # 2. Verificar que las carpetas y archivos existen
    rutas = [images_dir, csv_mapping_path, csv_typicality_path]
    for ruta in rutas:
        if not ruta.exists():
            print(f"❌ Error: No se encontró la ruta: {ruta.resolve()}")
            return  # <-- SOLUCIÓN 1: Usar 'return' en lugar de 'sys.exit(1)' para no romper Jupyter
            
    print("📂 Leyendo archivos CSV...")
    df_mapping = pd.read_csv(csv_mapping_path)
    df_typicality = pd.read_csv(csv_typicality_path)
    
    # --- MODO DEBUG: IMPRIMIR LAS COLUMNAS REALES ---
    print("\n🧐 Columnas reales en artikelnummer_to_image.csv:")
    print(df_mapping.columns.tolist())
    print("\n🧐 Columnas reales en article_typicality.csv:")
    print(df_typicality.columns.tolist())
    print("-" * 50)
    
    # ⚠️ CONFIGURACIÓN DE COLUMNAS
    # ¡Copia y pega los nombres EXACTOS de la lista que te imprima arriba!
    col_id_mapping = "artikelnummer"      
    col_img_mapping = "image_name"        
    col_id_typicality = "artikelnummer"
    
    # Validar columnas
    if col_id_mapping not in df_mapping.columns or col_img_mapping not in df_mapping.columns:
        print(f"❌ Error: Revisa las columnas de {csv_mapping_path.name}")
        return
    if col_id_typicality not in df_typicality.columns:
        print(f"❌ Error: Revisa las columnas de {csv_typicality_path.name}")
        return

    print("\n" + "="*50)
    print("🔍 1. ANÁLISIS CRUZADO DE CSVs (IDs)")
    print("="*50)
    
    # <-- SOLUCIÓN 2: Eliminar de raíz cualquier fila donde el ID o el nombre de imagen estén vacíos (NaN)
    df_mapping = df_mapping.dropna(subset=[col_id_mapping, col_img_mapping])
    df_typicality = df_typicality.dropna(subset=[col_id_typicality])
    
    ids_mapping = set(df_mapping[col_id_mapping].astype(str))
    ids_typicality = set(df_typicality[col_id_typicality].astype(str))
    
    solo_en_mapping = ids_mapping - ids_typicality
    solo_en_typicality = ids_typicality - ids_mapping
    
    print(f"IDs en artikelnummer_to_image : {len(ids_mapping)}")
    print(f"IDs en article_typicality     : {len(ids_typicality)}")
    print(f"⚠️ IDs SOLO en mapping          : {len(solo_en_mapping)}")
    print(f"⚠️ IDs SOLO en typicality       : {len(solo_en_typicality)}")

    print("\n" + "="*50)
    print("📸 2. ANÁLISIS DE IMÁGENES (CSV vs DISCO)")
    print("="*50)

    imagenes_esperadas = set()
    
    # Extraer imágenes con una segunda capa de seguridad contra celdas en blanco o textos 'nan'
    for img in df_mapping[col_img_mapping].unique():
        img_str = str(img).strip()
        
        # Ignorar si es una cadena vacía o el texto literal "nan"
        if not img_str or img_str.lower() == 'nan':
            continue
            
        if not img_str.lower().endswith(('.jpg', '.jpeg', '.png')):
            img_str = f"{img_str}.jpg"
        imagenes_esperadas.add(img_str)

    # Buscar imágenes físicamente en el disco
    imagenes_en_disco = set(
        f.name for f in images_dir.iterdir() if f.is_file() and f.suffix.lower() in ['.jpg', '.jpeg', '.png']
    )

    # Matemáticas de conjuntos para sacar diferencias
    faltantes_en_disco = imagenes_esperadas - imagenes_en_disco
    sobrantes_en_disco = imagenes_en_disco - imagenes_esperadas
    imagenes_correctas = imagenes_esperadas.intersection(imagenes_en_disco)

    print(f"Imágenes esperadas (según CSV)   : {len(imagenes_esperadas)}")
    print(f"Imágenes encontradas en disco    : {len(imagenes_en_disco)}")
    print(f"✅ Imágenes emparejadas con éxito : {len(imagenes_correctas)}")
    print(f"❌ Faltan en disco (están en CSV) : {len(faltantes_en_disco)}")
    print(f"❓ Sobran en disco (no en el CSV) : {len(sobrantes_en_disco)}")

    # 3. Exportar reportes
    if faltantes_en_disco or sobrantes_en_disco or solo_en_mapping or solo_en_typicality:
        print("\n" + "="*50)
        print("💾 EXPORTANDO REPORTES DE ERRORES...")
        print("="*50)
        
        report_dir = base_dir / "debug" / "reportes_errores"
        report_dir.mkdir(exist_ok=True)
        
        if faltantes_en_disco:
            (report_dir / "1_imagenes_faltantes_en_disco.txt").write_text("\n".join(faltantes_en_disco))
        if sobrantes_en_disco:
            (report_dir / "2_imagenes_sobrantes_en_disco.txt").write_text("\n".join(sobrantes_en_disco))
        if solo_en_mapping:
            (report_dir / "3_ids_solo_en_csv_mapping.txt").write_text("\n".join(solo_en_mapping))
        if solo_en_typicality:
            (report_dir / "4_ids_solo_en_csv_typicality.txt").write_text("\n".join(solo_en_typicality))
            
        print(f"Reportes guardados en la carpeta: {report_dir.resolve()}")
    else:
        print("\n🎉 ¡Todo perfecto! Los CSVs coinciden entre sí y todas las imágenes están correctas.")

    return list(imagenes_correctas)


# Al ejecutar la función en tu celda actual, guarda el resultado en una variable:
lista_correctas = reporte_exhaustivo()

📂 Leyendo archivos CSV...

🧐 Columnas reales en artikelnummer_to_image.csv:
['artikelnummer', 'image_name']

🧐 Columnas reales en article_typicality.csv:
['artikelnummer', 'typicality', 'Unnamed: 2']
--------------------------------------------------

🔍 1. ANÁLISIS CRUZADO DE CSVs (IDs)
IDs en artikelnummer_to_image : 107634
IDs en article_typicality     : 38335
⚠️ IDs SOLO en mapping          : 69299
⚠️ IDs SOLO en typicality       : 0

📸 2. ANÁLISIS DE IMÁGENES (CSV vs DISCO)
Imágenes esperadas (según CSV)   : 107634
Imágenes encontradas en disco    : 79307
✅ Imágenes emparejadas con éxito : 79304
❌ Faltan en disco (están en CSV) : 28330
❓ Sobran en disco (no en el CSV) : 3

💾 EXPORTANDO REPORTES DE ERRORES...
Reportes guardados en la carpeta: /pfs/data6/home/tu/tu_tu/tu_zxoxe46/austria_data/debug/reportes_errores


# Database Inner Join Pipeline
We want to check that our CSV files are merged into a single consolidated dataset using a strict inner join on the verified product ID key. 

This processing block cleans structural anomalies, strips whitespace formatting, and builds the primary dataset linking image paths directly to product typicality scores.

### Key Implementation Details

* **Artifact Removal**: Drops automated formatting columns like `Unnamed: 2` often generated by Excel or Google Sheets exports.
* **Type Normalization**: Converts the key identifier `artikelnummer` explicitly into a stripped text string to avoid type mismatches during cross-referencing.
* **Strict Join Enforcement**: Executes an inner join merge (`how='inner'`) to ensure records missing from either file are automatically omitted from the downstream dataset.
* **Persistence Export**: Writes the cleaned, consolidated dataset to a new CSV file (`mapped_typicality.csv`) with index arrays deactivated for straightforward machine consumption.

### File Inputs and Outputs
* `data/artikelnummer_to_image.csv` ➔ Source image mapping.
* `data/article_typicality.csv` ➔ Source dataset for typicality indexes.
* `data/mapped_typicality.csv` ➔ Final unified master file.


In [4]:
import pandas as pd
from pathlib import Path

# 1. Rutas
base_dir = Path(".")
csv_mapping_path = base_dir / "data" / "artikelnummer_to_image.csv"
csv_typicality_path = base_dir / "data" / "article_typicality.csv"
output_path = base_dir / "data" / "mapped_typicality.csv"

# 2. Cargar CSVs
df_mapping = pd.read_csv(csv_mapping_path)
df_typicality = pd.read_csv(csv_typicality_path)

# 3. Eliminar la columna 'Unnamed: 2' si existe
if 'Unnamed: 2' in df_typicality.columns:
    df_typicality = df_typicality.drop(columns=['Unnamed: 2'])

# Limpiar filas con NaN en la clave de cruce
df_mapping = df_mapping.dropna(subset=['artikelnummer'])
df_typicality = df_typicality.dropna(subset=['artikelnummer'])

# Estandarizar 'artikelnummer' a texto sin espacios para asegurar un cruce perfecto
df_mapping['artikelnummer'] = df_mapping['artikelnummer'].astype(str).str.strip()
df_typicality['artikelnummer'] = df_typicality['artikelnummer'].astype(str).str.strip()

# 4. Inner Join por 'artikelnummer'
df_mapped_typicality = pd.merge(df_mapping, df_typicality, on='artikelnummer', how='inner')

# 5. Exportar a CSV
df_mapped_typicality.to_csv(output_path, index=False)

print(f"✅ Inner Join realizado correctamente.")
print(f"📊 Registros resultantes: {len(df_mapped_typicality)}")
print(f"💾 Archivo guardado en: {output_path.resolve()}")

# Mostrar primeras filas
df_mapped_typicality.head()

✅ Inner Join realizado correctamente.
📊 Registros resultantes: 38335
💾 Archivo guardado en: /pfs/data6/home/tu/tu_tu/tu_zxoxe46/austria_data/data/mapped_typicality.csv


,artikelnummer,image_name,typicality
0,6039991,6039991_1.jpg,0.847559
1,6062701,6062701_1.jpg,0.811893
2,6064801,6064801_1.jpg,0.744338
3,7142910,7142910_1.jpg,0.773741
4,6325134,6325134_1.jpg,0.897172


# Physical Storage Verification
We want to check that our CSV files are verified against physical storage by confirming that every registered image asset is present on disk before running inference.

This execution block scans the localized image directory, maps relational dataframe records to absolute disk states, and flags broken image file references.

### Key Implementation Details

* **Local Disk Inventorying**: Crawls the directory specified by `images_dir` using `pathlib` to generate an optimized, high-speed tracking set containing valid graphic files (`.jpg`, `.jpeg`, `.png`).
* **Filename Structural Normalization**: Evaluates filename formatting anomalies, purges leading/trailing spaces, isolates null entries (`NaN`), and appends a default `.jpg` file extension if missing.
* **Vectorized Existence Cross-Reference**: Employs pandas `.isin()` vector mapping to immediately verify whether the formatted filename payload exists inside the physical storage set.
* **Dataframe Diagnostic Tracking**: Flags unlinked rows dynamically via a boolean column (`existe_en_disco`) and outputs structured previews containing identifiers, original filenames, and typicality metrics for isolated error tracking.

### File Inputs and Process Variables
* `images/` ➔ Target storage path containing source product images.
* `df_mapped_typicality['image_name_formatted']` ➔ Structured string column containing standardized target filename keys.
* `df_mapped_typicality['existe_en_disco']` ➔ Functional boolean mapping column used to filter out untracked image instances.


In [5]:
# 1. Carpeta de imágenes
images_dir = base_dir / "images"

# 2. Conjunto de imágenes reales en disco
imagenes_en_disco = set(
    f.name for f in images_dir.iterdir() 
    if f.is_file() and f.suffix.lower() in ['.jpg', '.jpeg', '.png']
)

# 3. Función para normalizar nombres de archivo (añade extensión .jpg si no la tiene)
def normalizar_nombre_imagen(nombre):
    if pd.isna(nombre):
        return ""
    nombre_str = str(nombre).strip()
    if not nombre_str.lower().endswith(('.jpg', '.jpeg', '.png')):
        nombre_str = f"{nombre_str}.jpg"
    return nombre_str

# Aplicar la normalización
df_mapped_typicality['image_name_formatted'] = df_mapped_typicality['image_name'].apply(normalizar_nombre_imagen)

# 4. Verificar presencia en disco
df_mapped_typicality['existe_en_disco'] = df_mapped_typicality['image_name_formatted'].isin(imagenes_en_disco)

# 5. Resultados del análisis
total_registros = len(df_mapped_typicality)
encontradas = df_mapped_typicality['existe_en_disco'].sum()
faltantes = total_registros - encontradas

print("="*50)
print("🔎 VERIFICACIÓN DE IMÁGENES DEL CSV EN DISCO")
print("="*50)
print(f"Total de registros en mapped_typicality.csv : {total_registros}")
print(f"✅ Imágenes ENCONTRADAS en disco             : {encontradas}")
print(f"❌ Imágenes FALTANTES en disco               : {faltantes}")

# Mostrar las filas correspondientes a imágenes faltantes si las hubiera
if faltantes > 0:
    print("\n⚠️ Registros cuya imagen NO está en la carpeta 'images/':")
    df_faltantes = df_mapped_typicality[~df_mapped_typicality['existe_en_disco']]
    display(df_faltantes[['artikelnummer', 'image_name', 'typicality']])
else:
    print("\n🎉 ¡Perfecto! Todas las imágenes de mapped_typicality.csv están presentes en el dataset.")

🔎 VERIFICACIÓN DE IMÁGENES DEL CSV EN DISCO
Total de registros en mapped_typicality.csv : 38335
✅ Imágenes ENCONTRADAS en disco             : 28213
❌ Imágenes FALTANTES en disco               : 10122

⚠️ Registros cuya imagen NO está en la carpeta 'images/':


,artikelnummer,image_name,typicality
4,6325134,6325134_1.jpg,0.897172
8,6458766,6458766_1.jpg,0.813716
9,6471837,6471837_1.jpg,0.646875
10,6499893,6499893_1.jpg,0.836038
11,6499897,6499897_1.jpg,0.846073
...,...,...,...
38323,7579901,7579901_1.jpg,0.733258
38326,7579905,7579905_1.jpg,0.627404
38327,7580613,7580613_1.jpg,0.814013
38329,7580615,7580615_1.jpg,0.805762


# Evaluation Dataset Subsampling
We want to check that our CSV files are verified and then sample a clean subset of confirmed physical images to populate a local development testing directory.

This script validates memory state references, randomly picks an isolated subset without replication, and copies the valid media binaries to a new directory.

### Key Implementation Details

* **State Isolation Check**: Evaluates runtime environment locals to verify `lista_correctas` is initialized and fully populated, preventing silent runtime failures.
* **Bounded Random Assignment**: Establishes a maximum sampling window using a baseline ceiling (20 items) contrasted against total available array dimensions to safely extract unique images.
* **Storage Allocation Execution**: Validates target directory states using dynamic path handling and forces a non-crashing folder creation rule (`exist_ok=True`).
* **Metadata-Preserving Binaries Migration**: Loops through array slices to verify source existence before calling `shutil.copy2()`, which preserves critical file creation and modification timestamps during migration.

### File Inputs and Outputs
* `images/` ➔ Source directory containing the full dataset of validated product images.
* `samples/` ➔ Target subdirectory populated with the isolated test samples.
* `lista_correctas` ➔ Dynamic variable array populated by the upstream database check pipeline.


In [ ]:
# import random
# import shutil
# from pathlib import Path

# # 1. Configurar rutas
# base_dir = Path(".")
# origen_dir = base_dir / "images"
# destino_dir = base_dir / "samples"

# # 2. Verificar que existan imágenes correctas cargadas de la celda anterior
# if "lista_correctas" not in locals() or not lista_correctas:
#     print(
#         "❌ Error: Ejecuta primero la celda anterior para detectar las imágenes correctas."
#     )
# else:
#     # Definir cuántas imágenes tomar (máximo las disponibles si son menos de 20)
#     cantidad_a_muestrear = min(20, len(lista_correctas))

#     # 3. Seleccionar las imágenes de forma aleatoria sin repetir
#     imagenes_seleccionadas = random.sample(lista_correctas, cantidad_a_muestrear)

#     # 4. Crear la carpeta de destino si no existe
#     destino_dir.mkdir(exist_ok=True)

#     print(
#         f"📸 Seleccionando {cantidad_a_muestrear} imágenes aleatorias de las {len(lista_correctas)} correctas..."
#     )
#     print("-" * 50)

#     # 5. Copiar los archivos físicos
#     contador_copiadas = 0
#     for nombre_img in imagenes_seleccionadas:
#         archivo_origen = origen_dir / nombre_img
#         archivo_destino = destino_dir / nombre_img

#         if archivo_origen.exists():
#             shutil.copy2(archivo_origen, archivo_destino)
#             print(f"✅ Copiada con éxito: {nombre_img}")
#             contador_copiadas += 1
#         else:
#             print(f"⚠️ Alerta: El archivo {nombre_img} no se encontró en físico.")

#     print("-" * 50)
#     print(
#         f"🎉 Proceso terminado. Se descargaron {contador_copiadas} imágenes en la carpeta: {destino_dir.resolve()}"
#     )
